[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/19_Gas_Consumption.ipynb)

# DiveLab

## Notebook 19 — Gas Consumption as a Dynamic Resource

**Guiding question:** How does depth change gas consumption, and how can we model remaining gas as another system state?

We now connect three parts of DiveLab:

$$
\boxed{\text{pressure}}
\rightarrow
\boxed{\text{breathing demand}}
\rightarrow
\boxed{\text{gas consumption}}
$$

Gas is no longer just an external resource.

It becomes a dynamic state of the dive system.

## Learning objectives

By the end of this notebook, you will be able to:

- explain why gas consumption increases with ambient pressure;
- distinguish surface-equivalent consumption from actual gas use at depth;
- define SAC/RMV-style consumption models;
- compute gas use at different depths;
- model cylinder gas as a state variable;
- simulate gas remaining during a depth profile;
- study how breathing demand and depth interact;
- connect gas consumption to planning, constraints and dive-computer logic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Ambient pressure revisited

From Notebook 01:

$$
P(z)=P_0+\rho gz.
$$

For simple diving calculations, we often use the approximate rule:

$$
P_{\text{abs}} \approx 1 + \frac{z}{10}
$$

in bar when $z$ is in metres of seawater.

This approximation is useful for building intuition.

In [ ]:
def ambient_pressure_bar(depth_m):
    return 1.0 + depth_m/10.0

# 2. Why gas consumption rises with depth

Suppose the diver breathes a surface-equivalent volume flow:

$$
\dot V_S
$$

measured in litres per minute at 1 bar.

At depth, the regulator supplies gas approximately at ambient pressure.

To provide the same lung-volume change at a higher pressure, more surface-equivalent gas is required.

A simple model is:

$$
\boxed{
\dot V_{\text{gas}}
=
\dot V_S
\frac{P_{\text{amb}}}{P_0}
}
$$

If $P_0=1$ bar:

$$
\dot V_{\text{gas}}
=
\dot V_S P_{\text{amb,bar}}.
$$

So, approximately:

- surface: $1$ bar → $1\times$ surface-equivalent rate;
- 10 m: $2$ bar → $2\times$;
- 20 m: $3$ bar → $3\times$;
- 30 m: $4$ bar → $4\times$.

This is one of the most important consequences of pressure with depth.

In [ ]:
depths = np.array([0, 10, 20, 30, 40])
pressure_ratio = ambient_pressure_bar(depths)

for z, p in zip(depths, pressure_ratio):
    print(f"{z:2d} m -> {p:.1f} bar -> {p:.1f}x surface-equivalent gas use")

# 3. SAC and RMV

Two related concepts are commonly used.

### SAC — Surface Air Consumption

Often expressed as a cylinder-pressure drop per unit time normalized to surface pressure.

### RMV — Respiratory Minute Volume

A surface-equivalent volumetric breathing rate:

$$
\text{RMV}
\quad [\mathrm{L/min}]
$$

For modeling, RMV is convenient because it is independent of cylinder size.

In this notebook we use a surface-equivalent volumetric rate.

# 4. Simple gas-use equation

Let:

$$
q_S
$$

be the surface-equivalent breathing demand in L/min.

Then at depth:

$$
\boxed{
q(z)
=
q_S P_{\text{abs}}(z)
}
$$

when absolute pressure is measured relative to 1 bar at the surface.

In [ ]:
def gas_rate_lpm(depth_m, surface_rate_lpm):
    return surface_rate_lpm * ambient_pressure_bar(depth_m)

surface_rate = 18.0

for z in [0, 10, 20, 30]:
    print(
        f"{z:2d} m -> {gas_rate_lpm(z, surface_rate):.1f} L/min "
        f"for surface-equivalent rate {surface_rate:.1f} L/min"
    )

# 5. Plot consumption versus depth

In [ ]:
depth_grid = np.linspace(0, 40, 300)
rate = gas_rate_lpm(depth_grid, surface_rate)

plt.plot(depth_grid, rate)
plt.xlabel("Depth [m]")
plt.ylabel("Gas use [surface L/min]")
plt.title("Gas consumption increases with ambient pressure")
plt.grid(True)
plt.show()

In this simple model, gas use rises approximately linearly with depth because absolute pressure does.

# 6. Breathing demand is not constant

The surface-equivalent breathing demand itself can vary.

We can write:

$$
q_S=q_{\text{rest}}\,m(t)
$$

where $m(t)$ is a demand multiplier.

For example:

- calm/resting;
- moderate effort;
- high workload.

Then:

$$
q(z,t)
=
q_{\text{rest}}
m(t)
P_{\text{abs}}(z).
$$

In [ ]:
q_rest = 16.0

multipliers = {
    "calm": 1.0,
    "moderate": 1.5,
    "high demand": 2.5
}

for label, m in multipliers.items():
    print(
        f"{label:12s}: "
        f"{gas_rate_lpm(20, q_rest*m):.1f} L/min at 20 m"
    )

This shows two independent effects:

$$
\boxed{\text{depth}}
$$

and:

$$
\boxed{\text{breathing demand}}
$$

both multiply gas consumption.

# 7. Gas remaining as a state variable

Let:

$$
G(t)
$$

be remaining surface-equivalent gas volume.

Then:

$$
\boxed{
\dot G(t)=-q(z,t)
}
$$

This is a first-order resource-depletion state equation.

In state-space language, we can extend the dive state:

$$
x=
\begin{bmatrix}
z\\
v\\
G
\end{bmatrix}.
$$

Now the system contains both motion and resource state.

# 8. Simple constant-depth simulation

Suppose the diver remains at 20 m for 30 minutes.

Assume:

$$
q_S=18\ \mathrm{L/min}.
$$

In [ ]:
def simulate_gas_constant_depth(
    depth_m,
    duration_min,
    surface_rate_lpm,
    initial_gas_l
):
    dt = 0.01  # min
    t = np.arange(0, duration_min+dt, dt)

    G = np.zeros_like(t)
    G[0] = initial_gas_l

    rate = gas_rate_lpm(depth_m, surface_rate_lpm)

    for k in range(len(t)-1):
        G[k+1] = max(G[k] - rate*dt, 0.0)

    return t, G

t_g, G = simulate_gas_constant_depth(
    depth_m=20,
    duration_min=30,
    surface_rate_lpm=18,
    initial_gas_l=2400
)

In [ ]:
plt.plot(t_g, G)
plt.xlabel("Time [min]")
plt.ylabel("Remaining surface-equivalent gas [L]")
plt.title("Gas remaining at constant depth")
plt.grid(True)
plt.show()

At constant depth and constant breathing demand, remaining gas decreases approximately linearly with time.

# 9. Cylinder pressure as another representation

If a cylinder has internal water volume:

$$
V_c
$$

and pressure:

$$
P_c,
$$

a simple idealized surface-equivalent gas content is:

$$
G \approx V_c P_c
$$

when $V_c$ is in litres and $P_c$ in bar.

For example, a 12 L cylinder at 200 bar contains approximately:

$$
12\times200=2400
$$

surface litres in this simplified model.

In [ ]:
cylinder_volume_l = 12.0
pressure_bar = 200.0

gas_content = cylinder_volume_l * pressure_bar
print(f"Approximate gas content: {gas_content:.0f} surface litres")

This ignores real-gas effects and reserve/planning considerations.

It is a simple educational model.

# 10. Convert remaining gas to cylinder pressure

If:

$$
G=V_cP_c,
$$

then:

$$
\boxed{
P_c=\frac{G}{V_c}
}
$$

In [ ]:
pressure_remaining = G / cylinder_volume_l

plt.plot(t_g, pressure_remaining)
plt.xlabel("Time [min]")
plt.ylabel("Cylinder pressure [bar]")
plt.title("Idealized cylinder pressure at constant depth")
plt.grid(True)
plt.show()

# 11. Dynamic depth profile

Now let depth vary over time.

We define a simple profile:

- descent;
- bottom segment;
- ascent.

In [ ]:
def depth_profile(t_min):
    if t_min < 3:
        return 20/3 * t_min

    if t_min < 18:
        return 20.0

    if t_min < 22:
        return 20 - 5*(t_min-18)

    return 0.0

In [ ]:
t_profile = np.linspace(0, 25, 1000)
z_profile = np.array([depth_profile(tt) for tt in t_profile])

plt.plot(t_profile, z_profile)
plt.gca().invert_yaxis()
plt.xlabel("Time [min]")
plt.ylabel("Depth [m]")
plt.title("Example dive profile")
plt.grid(True)
plt.show()

Depth changes ambient pressure, so gas-use rate changes throughout the dive.

In [ ]:
rate_profile = gas_rate_lpm(z_profile, surface_rate)

plt.plot(t_profile, rate_profile)
plt.xlabel("Time [min]")
plt.ylabel("Gas use [surface L/min]")
plt.title("Gas-use rate along the dive profile")
plt.grid(True)
plt.show()

# 12. Integrate gas use along the profile

In [ ]:
def simulate_gas_profile(
    duration_min=25,
    dt=0.01,
    surface_rate_lpm=18,
    initial_gas_l=2400,
    demand_fn=None
):
    t = np.arange(0, duration_min+dt, dt)

    z = np.zeros_like(t)
    q = np.zeros_like(t)
    G = np.zeros_like(t)

    G[0] = initial_gas_l

    for k in range(len(t)-1):
        z[k] = depth_profile(t[k])

        demand_multiplier = 1.0 if demand_fn is None else demand_fn(t[k])

        q[k] = gas_rate_lpm(
            z[k],
            surface_rate_lpm*demand_multiplier
        )

        G[k+1] = max(G[k] - q[k]*dt, 0.0)

    z[-1] = depth_profile(t[-1])
    q[-1] = q[-2]

    return t, z, q, G

In [ ]:
t, z, q, G = simulate_gas_profile()

plt.plot(t, G)
plt.xlabel("Time [min]")
plt.ylabel("Remaining gas [surface L]")
plt.title("Gas remaining during a dive profile")
plt.grid(True)
plt.show()

# 13. Gas consumption is state-dependent

The gas depletion equation is:

$$
\dot G
=
-q_S P_{\text{abs}}(z).
$$

So gas depletion depends on another state:

$$
z(t).
$$

This makes gas consumption a **state-coupled resource dynamic**.

# 14. Add changing workload

Suppose breathing demand rises temporarily.

For example, define a multiplier:

$$
m(t)>1.
$$

In [ ]:
def workload_multiplier(t_min):
    if 10 <= t_min <= 14:
        return 2.0
    return 1.0

In [ ]:
t_w, z_w, q_w, G_w = simulate_gas_profile(
    demand_fn=workload_multiplier
)

plt.plot(t, G, label="Baseline demand")
plt.plot(t_w, G_w, label="Temporary higher demand")

plt.xlabel("Time [min]")
plt.ylabel("Remaining gas [surface L]")
plt.title("Effect of breathing demand on gas remaining")
plt.grid(True)
plt.legend()
plt.show()

Depth and workload interact multiplicatively.

Higher demand at depth consumes much more gas than the same increase near the surface.

# 15. Compare the same high-demand interval at different depths

Suppose the surface-equivalent rate doubles from:

$$
18\to36\ \mathrm{L/min}.
$$

The additional gas use is:

$$
18P_{\text{abs}}(z)
$$

surface L/min.

So the same extra breathing demand costs more gas at greater depth.

In [ ]:
for depth in [0, 10, 20, 30]:
    baseline = gas_rate_lpm(depth, 18)
    high = gas_rate_lpm(depth, 36)

    print(
        f"{depth:2d} m -> additional gas use = {high-baseline:.1f} L/min"
    )

# 16. Connection to breathing dynamics

Notebook 18 modeled breathing as part of buoyancy control.

Now we see that breathing also affects resource consumption.

So breathing has at least three systems roles:

$$
\boxed{\text{buoyancy disturbance}}
$$

$$
\boxed{\text{buoyancy control}}
$$

$$
\boxed{\text{gas-consumption driver}}
$$

This creates a control tradeoff.

A breathing pattern that changes buoyancy also changes gas demand.

In a more complete model, control, physiology and resource use are coupled.

# 17. Gas as a constraint

Gas remaining can be treated as a constraint:

$$
G(t)\ge G_{\min}.
$$

This connects naturally to Model Predictive Control from Notebook 12.

A predictive controller or planner could ask:

> Is this depth/time trajectory feasible given the available gas resource?

In optimization language, one might include:

$$
G(t_f)\ge G_{\text{reserve}}
$$

as a terminal constraint.

This is not a dive-planning recommendation here; it is a control-theory example of a finite resource constraint.

# 18. Time-to-empty estimate

At constant depth and constant breathing demand:

$$
T_{\text{empty}}
=
\frac{G}{q}.
$$

In [ ]:
def time_to_empty_min(
    gas_remaining_l,
    depth_m,
    surface_rate_lpm
):
    q = gas_rate_lpm(depth_m, surface_rate_lpm)
    return gas_remaining_l/q

for depth in [0, 10, 20, 30]:
    T = time_to_empty_min(
        gas_remaining_l=1200,
        depth_m=depth,
        surface_rate_lpm=18
    )
    print(f"{depth:2d} m -> {T:.1f} min in this simple constant-condition model")

This number is only meaningful under the assumptions:

- constant depth;
- constant breathing demand;
- simple gas model.

Real conditions change continuously.

# 19. Why a dive computer cannot know future gas use exactly

Even if a system knows current:

- depth;
- cylinder pressure;
- recent breathing rate;

future gas use depends on:

- future depth;
- future workload;
- disturbances;
- diver behavior.

So gas prediction is a **forecasting problem**.

This is similar to state estimation:

$$
\text{current measurements}
+
\text{model}
\rightarrow
\text{future estimate}.
$$

Notebook 20 will use this idea when we examine dive computers as measurement-and-model systems.

# 20. A dynamic state-space view

A simplified state could be:

$$
x=
\begin{bmatrix}
z\\
v\\
G
\end{bmatrix}.
$$

Then:

$$
\dot z=-v
$$

$$
\dot v=f(z,v,u)
$$

$$
\dot G
=
-q_S(t)P_{\text{abs}}(z).
$$

The resource state $G$ does not directly cause vertical motion in this simplified model, but vertical motion changes its depletion rate.

This is a **one-way coupling**:

$$
z
\rightarrow
q
\rightarrow
G.
$$

A more complete model could also include effects of gas mass and equipment changes, but those are beyond this notebook.

# 21. Depth-time exposure

Gas consumption depends on the integral:

$$
G_{\text{used}}
=
\int_0^T
q_S(t)P_{\text{abs}}(z(t))\,dt.
$$

So gas use depends on the entire trajectory, not just maximum depth.

This is a recurring systems idea:

> cumulative effects depend on the history of the state.

We saw the same structure with:

- integral control;
- observer history;
- decompression models, which will come later.

# 22. Compare two profiles with the same maximum depth

Profile A:

- short bottom time.

Profile B:

- longer bottom time.

Same maximum depth does not imply the same gas use.

In [ ]:
def make_profile(bottom_time):
    def profile(t):
        if t < 3:
            return 20/3*t

        if t < 3 + bottom_time:
            return 20.0

        ascent_start = 3 + bottom_time

        if t < ascent_start + 4:
            return 20 - 5*(t-ascent_start)

        return 0.0

    return profile

In [ ]:
def integrate_profile(profile_fn, duration, surface_rate=18, dt=0.01):
    t = np.arange(0, duration+dt, dt)
    G_used = 0.0

    for k in range(len(t)-1):
        z = profile_fn(t[k])
        q = gas_rate_lpm(z, surface_rate)
        G_used += q*dt

    return G_used

profile_A = make_profile(10)
profile_B = make_profile(20)

used_A = integrate_profile(profile_A, 20)
used_B = integrate_profile(profile_B, 30)

print(f"Profile A gas used: {used_A:.0f} surface L")
print(f"Profile B gas used: {used_B:.0f} surface L")

# 23. Gas consumption and optimization

Suppose a trajectory planner wants to minimize a cost:

$$
J
=
w_t T
+
w_g G_{\text{used}}
+
w_u U_{\text{effort}}.
$$

Now time, gas and control effort all compete.

This is the kind of multi-objective problem that advanced planning algorithms can address.

# 24. Important modeling limitations

This notebook uses deliberately simple assumptions.

It ignores, among other things:

- real-gas behavior at high cylinder pressure;
- regulator characteristics;
- gas temperature;
- changing workload physiology;
- dead space and ventilation efficiency;
- gas-sharing scenarios;
- reserve-planning procedures;
- equipment-specific gas use.

The purpose is to expose the dynamical structure, not to create an operational gas-planning tool.

# Exercises

### 1. Consumption by depth

For surface-equivalent breathing rates:

$$
12,\ 18,\ 25\ \mathrm{L/min},
$$

compute gas use at:

$$
0,\ 10,\ 20,\ 30\ \mathrm{m}.
$$

In [ ]:
# Your code here

### 2. Cylinder comparison

Using the simple idealized model:

$$
G=V_cP_c,
$$

compare:

- 10 L at 200 bar;
- 12 L at 200 bar;
- 15 L at 200 bar.

In [ ]:
# Your code here

### 3. Variable breathing demand

Create a demand multiplier that rises gradually rather than suddenly.

How does gas remaining change?

In [ ]:
# Your code here

### 4. Two dive profiles

Construct two profiles with the same maximum depth but different depth-time histories.

Compare total gas use.

In [ ]:
# Your code here

### 5. Gas-state simulation

Extend the state:

$$
x=
[z,\ v,\ G]^T
$$

and simulate motion and gas consumption together.

In [ ]:
# Your code here

# Challenge — predictive gas estimate

At each point in a dive profile:

1. measure current depth;
2. estimate current surface-equivalent breathing rate;
3. predict gas use for the next 5 minutes if current conditions continue;
4. compare that forecast with what actually happens when the profile changes.

This illustrates the difference between:

$$
\boxed{\text{state estimate}}
$$

and:

$$
\boxed{\text{future prediction}}.
$$

In [ ]:
# Your code here

# Summary

Gas consumption can be modeled as another dynamical state.

A simple relationship is:

$$
q(z,t)
=
q_S(t)P_{\text{abs}}(z).
$$

Remaining gas satisfies:

$$
\boxed{
\dot G=-q(z,t)
}
$$

We learned that:

- depth increases gas consumption through ambient pressure;
- breathing demand and depth multiply;
- remaining gas is a resource state;
- cylinder pressure can be related to gas content in a simplified model;
- gas use depends on the complete depth-time trajectory;
- future gas use is a prediction problem;
- gas constraints connect naturally to trajectory planning and MPC.

Breathing now has three roles in DiveLab:

$$
\boxed{
\text{buoyancy disturbance}
}
$$

$$
\boxed{
\text{buoyancy control}
}
$$

$$
\boxed{
\text{resource consumption}
}
$$

### Next — Notebook 20

The next roadmap topic is **Dive Computers**.

We can study a dive computer as a systems architecture:

$$
\boxed{
\text{pressure sensor}
\rightarrow
\text{depth estimate}
\rightarrow
\text{time-depth history}
\rightarrow
\text{models}
\rightarrow
\text{displayed information}
}
$$

This will connect sensors, estimation, numerical integration, gas tracking and eventually decompression models.